<a href="https://colab.research.google.com/github/Junhojuno/pytorch-tutorial/blob/master/Make_ResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ResNet의 코드를 해부해보자
- Block은 BasicBlock과 Bottleneck으로 나뉜다.
- downsample은 stride가 2일때 output의 size가 줄어들어 identity(input_data)와 size가 맞지않게되는것을 해소하기위해 해주는 것이다.(Block들의 '\__init__'부분에 있는 downsample의 경우를 말함.)
- 코드 중간에 downsample이 실행되는부분이 있는데 , 이는 channels를 맞춰주기 위함이다.(_make_layer()내에 있는 downsample의 경우를 말함)

In [0]:
import torch.nn as nn
import torch.utils.model_zoo as model_zoo


__all__ = ['ResNet', 'resnet18', 'resnet34', 'resnet50', 'resnet101',
           'resnet152']


model_urls = {
    'resnet18': 'https://download.pytorch.org/models/resnet18-5c106cde.pth',
    'resnet34': 'https://download.pytorch.org/models/resnet34-333f7ec4.pth',
    'resnet50': 'https://download.pytorch.org/models/resnet50-19c8e357.pth',
    'resnet101': 'https://download.pytorch.org/models/resnet101-5d3b4d8f.pth',
    'resnet152': 'https://download.pytorch.org/models/resnet152-b121ed2d.pth',
}

In [0]:
# 3x3, 1x1 convolution을 사용한다.

def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False) # padding은 1이구요


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False) # 여긴 padding을 사용하지 않습니다.(default 0)

### BasicBlock
![BasicBlock](./asset/(ResNet)BasicBlock.PNG)


In [0]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        # 만약 input x의 shape이  3x64x64라면
        identity = x
        # identity shape : 3x64x64
  
        out = self.conv1(x) # 3x3 convolution에 별도 stride를 입력받도록함.(init에서 받은게 적용됨)
        # 만약 stride가 2였다면,
        # out shape : 3x32x32
        out = self.bn1(out)
        out = self.relu(out)
        # out shape : 3x32x32
        out = self.conv2(out) # 3x3 convolution에 stride=1고정
        out = self.bn2(out)
        # out shape : 3x32x32
        # but identity shape : 3x64x64
        # size가 맞지 않는다... 이런경우 downsample이 필요
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

### Bottleneck (block)
![Bottleneck](./asset/(ResNet)Bottleneck.PNG)

In [0]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = conv1x1(inplanes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion) # channel이 4배가 점프되는 걸 expansion으로 곱해주는걸로 표현
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x) # 1x1 stride=1
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out) # 3x3 stride는 init의 stride
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out) # 1x1 stride=1
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

In [0]:
# model = ResNet(Bottleneck, [3, 4, 6, 3], **kwargs) --> ResNet50을 예로들어 설명하도록한다.
class ResNet(nn.Module):

    def __init__(self, block, layers, num_classes=1000, zero_init_residual=False):
        super(ResNet, self).__init__()
        
        self.inplanes = 64
        
        # 여기서 input은 3x224x224이다.
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # output = self.conv1(input)
        # output shape : 64x112x112
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        
        # input : 64x112x112
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        # output : 64x56x56
        
        # 여기서부터는 아래 _make_layer()로 가세요.
        self.layer1 = self._make_layer(block, 64, layers[0]) # layers[0] = 3
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2) # layers[1] = 4
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2) # layers[2] = 6 
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2) # layers[3] = 3
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)
                    
    # self.inplanes = 64
    # self.layer1 = self._make_layer(Bottleneck, 64, 3)
    # ---------------------------------------------------
    # layer2로 넘어가면서, self.inplanes = 256
    # self.layer2 = self._make_layer(Bottleneck, 128, 4, stride=2)
    # layer2는 바로 layers = [] 부분으로 넘어가세요.(잔계산은 하지 않겠습니다.)
    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion: # self.inplanes = 64, planes*block.expansion= 64*4, 둘이 같지않음. (if문 안으로 들어간다.)
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride), # conv1x1(64, 64*4, stride=1)
                nn.BatchNorm2d(planes * block.expansion), # BatchNorm2d(256)
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        # self.layer1 : layers.append(Bottleneck(64, 64, 1, downsample))
        # self.layer2 : layers.append(Bottleneck(256, 128, 2, downsample))
  
        self.inplanes = planes * block.expansion # self.inplanes = 256
        for _ in range(1, blocks): # range(1,3), ;blocks=3이었다.
            layers.append(block(self.inplanes, planes)) # 2번 append

        return nn.Sequential(*layers)
      
      #결국 self.layer1 = nn.Sequential[Bottleneck(64, 64, 1, downsample),
      #                                ,Bottleneck(256, 64)
      #                                ,Bottleneck(256, 64)]
      
      #결국 self.layer2 = nn.Sequential[Bottleneck(256, 128, 2, downsample),
      #                                ,Bottleneck(512, 128)
      #                                ,Bottleneck(512, 128)]
      #                                ,Bottleneck(512, 128)]

      

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [0]:

def resnet18(pretrained=False, **kwargs):
    """Constructs a ResNet-18 model.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(BasicBlock, [2, 2, 2, 2], **kwargs) # 시작단의 conv 1개와 BasicBlock으로 기본생성되는 convolution layer 2개에 [2,2,2,2]각각 2개씩 conv를 만들기 때문에 2x(2+2+2+2), 여기에 fc layer 1 = 18개
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls['resnet18']))
    return model

In [0]:
def resnet50(pretrained=False, **kwargs):
    """Constructs a ResNet-50 model.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(Bottleneck, [3, 4, 6, 3], **kwargs) # 시작단의 conv 1개와 Bottleneck으로 기본생성되는 convolution layer 3개에 [3,4,6,3]각각 3,4,6,3개씩 conv를 만들기 때문에 3x(3+4+6+3), 여기에 fc layer 1 = 50개
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls['resnet50']))
    return model

In [6]:
# 실제 위에서 예제로 실행한 layer1부분과 비교해보세요
resnet = resnet50()
resnet

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=F

### Let's Play ResNet
- VGG와 모델을 제외한 전체적인 코드는 동일
- 데이터는 CIFAR10